# T35 -- Phase 3: AIS evidence hook on real data

**No AIS data exists anywhere in this project** (`data/` contains only `ocean.nc`,
`wind.nc`, `database_payload.json`). This notebook confirms the pipeline runs
end-to-end with AIS absent (every real candidate correctly resolves to
`ais_available=False`, never a crash or fabricated value), and demonstrates the
`renormalize_weighted_mean` utility's behavior both with and without the AIS
component present (the "present" case uses a SYNTHETIC test-double value only, never
real AIS evidence, which does not exist).

In [1]:
import os, sys, json
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np

from iste.candidate_extraction import Candidate, CoreRegion
from iste.phase3_ais import AISDataSource, compute_ais_evidence, renormalize_weighted_mean

## 1. Confirm no AIS data exists in this project

In [2]:
data_files = sorted(os.listdir(os.path.join(REPO_ROOT, "data")))
print(f"data/ contents: {data_files}")
assert not any("ais" in f.lower() or "vessel" in f.lower() for f in data_files)
print("confirmed: no AIS/vessel data file present")

data/ contents: ['database_payload.json', 'ocean.nc', 'wind.nc']
confirmed: no AIS/vessel data file present


## 2. Run the AIS hook over all 280 real candidates -- confirm zero crashes, zero fabricated values

In [3]:
hotspots_path = os.path.join(REPO_ROOT, "results", "actual_data", "hotspots.json")
hotspots = json.load(open(hotspots_path, encoding="utf-8"))
print(f"loaded {len(hotspots)} real candidates from {hotspots_path}")

real_candidates = []
for h in hotspots:
    real_candidates.append(Candidate(
        candidate_id=h["candidate_id"], x_peak=h["lon"], y_peak=h["lat"],
        t_s=np.datetime64(h["t_s"], "ns"),
        core_region=CoreRegion(center_lat=h["lat"], center_lon=h["lon"], radius_m=1000.0),
        support_score=h["support_score"], density_score=h["density_score"],
        weighted_support_score=h["weighted_support_score"], pixel_diversity_score=h["pixel_diversity_score"],
        peak_support_features=None,
    ))

ais_results = [compute_ais_evidence(c) for c in real_candidates]
print(f"ran compute_ais_evidence over {len(ais_results)} real candidates -- zero exceptions raised")

n_available = sum(1 for r in ais_results if r.ais_available)
n_unavailable = sum(1 for r in ais_results if not r.ais_available)
n_score_none = sum(1 for r in ais_results if r.ais_score is None)
print(f"ais_available=True:  {n_available}")
print(f"ais_available=False: {n_unavailable}")
print(f"ais_score is None:   {n_score_none} (must equal total candidate count -- never a fabricated 0.0)")
assert n_available == 0 and n_unavailable == len(real_candidates) and n_score_none == len(real_candidates)
print()
print("sample reason (identical for every real candidate):")
print(f"  {ais_results[0].reason}")

loaded 280 real candidates from C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\results\actual_data\hotspots.json
ran compute_ais_evidence over 280 real candidates -- zero exceptions raised
ais_available=True:  0
ais_available=False: 280
ais_score is None:   280 (must equal total candidate count -- never a fabricated 0.0)

sample reason (identical for every real candidate):
  AIS unavailable: no AIS data source is configured or exists in this project (data/ contains only ocean.nc, wind.nc, database_payload.json)


## 3. Confirm the abstract interface cannot be silently used as if implemented

In [4]:
try:
    AISDataSource()
    print("FAIL: expected TypeError")
except TypeError as e:
    print(f"correctly cannot instantiate the bare abstract interface: {e}")

class ExampleFutureSource(AISDataSource):
    def query_vessels_near(self, lat, lon, t, radius_m, time_window_hours):
        return []

try:
    compute_ais_evidence(real_candidates[0], ais_source=ExampleFutureSource())
    print("FAIL: expected NotImplementedError")
except NotImplementedError as e:
    print(f"correctly raises NotImplementedError if a source were ever supplied (no real formula exists): {e}")

correctly cannot instantiate the bare abstract interface: Can't instantiate abstract class AISDataSource without an implementation for abstract method 'query_vessels_near'
correctly raises NotImplementedError if a source were ever supplied (no real formula exists): AIS-present evidence computation is not implemented: no real AIS data exists in this project to design or validate a combination formula against. Implement this path only once a concrete AISDataSource backed by real data is available.


## 4. `renormalize_weighted_mean`: with and without AIS present

Demonstrates that a downstream composite score (T36's future job) is well-defined in
BOTH cases. The "AIS present" case uses a SYNTHETIC placeholder value purely to
exercise the renormalization mechanics -- it is explicitly NOT real AIS evidence.

In [5]:
# stand-in values for 5 candidates' worth of OTHER Phase 3 evidence components
# (illustrative synthetic numbers, not read from any real T30-T34 output here --
# purely to demonstrate the renormalization utility's mechanics)
example_components_ais_absent = [
    (1.0, 0.62),  # e.g. normalized_kde-like component
    (1.0, 0.41),  # e.g. persistence-like component
    (1.0, 0.77),  # e.g. convergence-like component
    (1.0, 0.55),  # e.g. dispersion-aware-like component
    (1.0, 0.70),  # e.g. screening-like component
    (1.0, None),  # AIS: unavailable in this project -> None
]
score_ais_absent = renormalize_weighted_mean(example_components_ais_absent)

SYNTHETIC_AIS_TEST_DOUBLE = 0.35  # NOT real AIS evidence -- illustrative only
example_components_ais_present = example_components_ais_absent[:-1] + [(1.0, SYNTHETIC_AIS_TEST_DOUBLE)]
score_ais_present = renormalize_weighted_mean(example_components_ais_present)

print(f"composite score with AIS ABSENT (renormalized over 5 available components): {score_ais_absent:.4f}")
print(f"composite score with AIS PRESENT (synthetic test double, 6 components):      {score_ais_present:.4f}")
print()
print("both well-defined (no NaN, no crash) -- confirms the renormalization mechanics T36 will rely on.")
assert not np.isnan(score_ais_absent) and not np.isnan(score_ais_present)

composite score with AIS ABSENT (renormalized over 5 available components): 0.6100
composite score with AIS PRESENT (synthetic test double, 6 components):      0.5667

both well-defined (no NaN, no crash) -- confirms the renormalization mechanics T36 will rely on.


## Known limitations (T35 scope only)

- No AIS data exists in this project; the AIS-present code path
  (`compute_ais_evidence` with a real `AISDataSource`) raises `NotImplementedError` by
  design -- implementing a real scoring formula now, with nothing to validate it
  against, would mean inventing an untested observation model.
- `renormalize_weighted_mean` is a general utility, not AIS-specific; its "AIS present"
  demonstration above uses a synthetic placeholder value, never real evidence.
- No Stage III module, and no existing T14-T17/T25/T26/T29-T34 module, was modified.